In [8]:
import os
cwd = os.getcwd()   
print(f"Current working directory: {cwd}")

Current working directory: /home/houxuc/snap/Topt


In [4]:
import os
import torch
import pickle
from transformers import AutoTokenizer, AutoModelForMaskedLM
from utils.foldseek_util import get_struc_seq

MODEL_ID = "westlake-repl/SaProt_650M_PDB"

INPUT_DIR = "/home/houxuc/snap/Topt/structures"
OUTPUT_DIR = "/home/houxuc/snap/Topt/embeddings/saprot_650m_features"
FOLDSEEK_BIN = "/home/houxuc/snap/Topt/bin/foldseek"

os.makedirs(OUTPUT_DIR, exist_ok=True)

if torch.cuda.is_available():
    device = "cuda" if torch.cuda.device_count() > 1 else "cuda:0"
else:
    device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForMaskedLM.from_pretrained(MODEL_ID)
model.to(device)
model.eval()

structure_files = [
    os.path.join(INPUT_DIR, f)
    for f in os.listdir(INPUT_DIR)
    if f.endswith((".pdb", ".cif"))
]

print(f"Found {len(structure_files)} structure files.")

for pdb_path in structure_files:
    uniprot_id = os.path.splitext(os.path.basename(pdb_path))[0]

    try:
        parsed_seqs = get_struc_seq(FOLDSEEK_BIN, pdb_path, ["A"], plddt_mask=False)

        if "A" not in parsed_seqs:
            print(f"Chain A not found: {pdb_path}")
            continue

        seq, foldseek_seq, combined_seq = parsed_seqs["A"]

        inputs = tokenizer(
            combined_seq,
            return_tensors="pt",
            truncation=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            residue_embeddings = outputs.hidden_states[-1][0, 1:-1, :]   # [1, L, D]

        save_path = os.path.join(OUTPUT_DIR, f"{uniprot_id}.pkl")
        with open(save_path, "wb") as outfile:
            pickle.dump(residue_embeddings.detach().cpu(), outfile)

        print(f"Saved: {save_path} | shape: {tuple(residue_embeddings[0].shape)}")

    except Exception as e:
        print(f"Error processing {pdb_path}: {e}")

Some weights of EsmForMaskedLM were not initialized from the model checkpoint at westlake-repl/SaProt_650M_PDB and are newly initialized: ['esm.contact_head.regression.weight', 'esm.contact_head.regression.bias', 'esm.embeddings.position_embeddings.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Found 3228 structure files.
Saved: /home/houxuc/snap/Topt/embeddings/saprot_650m_features/A0A0H3NGZ8.pkl | shape: (1280,)
Saved: /home/houxuc/snap/Topt/embeddings/saprot_650m_features/P75785.pkl | shape: (1280,)
Saved: /home/houxuc/snap/Topt/embeddings/saprot_650m_features/Q97U35.pkl | shape: (1280,)
Saved: /home/houxuc/snap/Topt/embeddings/saprot_650m_features/Q70FD1.pkl | shape: (1280,)
Saved: /home/houxuc/snap/Topt/embeddings/saprot_650m_features/Q9XHE2.pkl | shape: (1280,)
Saved: /home/houxuc/snap/Topt/embeddings/saprot_650m_features/Q6XMT2.pkl | shape: (1280,)
Saved: /home/houxuc/snap/Topt/embeddings/saprot_650m_features/B9A1J7.pkl | shape: (1280,)
Saved: /home/houxuc/snap/Topt/embeddings/saprot_650m_features/A4VLZ3.pkl | shape: (1280,)
Saved: /home/houxuc/snap/Topt/embeddings/saprot_650m_features/Q25BT4.pkl | shape: (1280,)
Saved: /home/houxuc/snap/Topt/embeddings/saprot_650m_features/Q9WZ19.pkl | shape: (1280,)
Saved: /home/houxuc/snap/Topt/embeddings/saprot_650m_features/D2E4A5